# MMA3001 – 3D apple-tree skeleton from RGB-D images (Colab runner)
This notebook is a thin front-end: **all code lives in the GitHub repository** (`src/treeskel`). It installs the package, trains/evaluates the models, shows the results and runs the final
`extract_skeleton_3d` demo. Use *Runtime → Change runtime type → GPU (T4)* for the CNN.

1. Edit `REPO_URL` and `CFG_OVERRIDES` below.  2. *Runtime → Run all.*  The labelled TreeID data are bundled in the repo (`data/TreeID`), so no Drive mount or download is needed: leave `DATA_MODE="repo"`. Use `"synthetic"` + `QUICK_TEST=True` only for a fast smoke test.

In [ ]:
REPO_URL = "https://github.com/abirkbexk/tree-skeleton-3d.git"     # <-- change
CFG_OVERRIDES = dict(
    DATA_MODE="repo",          # 'repo' (bundled data/TreeID) | 'gdown' | 'drive' | 'local' | 'synthetic'
    QUICK_TEST=False,          # True = tiny/fast smoke test
    RUN_CNN=False,             # True needs PyTorch (preinstalled on Colab)
    # LOCAL_PATH="/content/drive/MyDrive/MMA3001/dataset1", GROUP_REGEX=None, SKELETON_VARIANT=None,
)

In [ ]:
import os, sys, subprocess
if not os.path.exists("tree-skeleton-3d"):
    subprocess.run(["git", "clone", REPO_URL], check=True)
%cd tree-skeleton-3d
!pip -q install -e .

In [ ]:
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from treeskel import get_config, train_pipeline, extract_skeleton_3d
cfg = get_config(**CFG_OVERRIDES)
out = train_pipeline(cfg, out_dir="outputs")            # selects the model on VALIDATION data, writes outputs/skeleton_model.joblib
ds, X = out["ds"], out["X"]; TR, VA, TE = out["splits"]
out["results"]

## Model comparison and post-processing choice

In [ ]:
res = out["results"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
res[["val_rmse_px", "test_rmse_px"]].plot.bar(ax=ax[0]); ax[0].set_ylabel("RMSE (px)")
ax[1].scatter(res.predict_ms_per_img, res.test_rmse_px)
for n, r in res.iterrows(): ax[1].annotate(n, (r.predict_ms_per_img, r.test_rmse_px), fontsize=8)
ax[1].set_xscale("log"); ax[1].set_xlabel("ms / image"); ax[1].set_ylabel("test RMSE (px)"); plt.tight_layout(); plt.show()
print("selected post-processing:", out["post"]); out["post_table"].round(3)

## Predictions vs ground truth on test images (worst / median / best)

In [ ]:
from treeskel.metrics import evaluate_predictions
best = res.index[0]; pred = out["model"].predict(ds["rgb"][TE], ds["dnorm"][TE])[0]
m = evaluate_predictions(pred, ds["rows"][TE], ds["valid"][TE]); order = np.argsort(-m.rmse_px.values); sel = [order[0], order[len(order)//2], order[-1]]
fig, ax = plt.subplots(1, 3, figsize=(10, 3.6)); ys = np.arange(cfg["IMG_SIZE"])
for a, j in zip(ax, sel):
    i = TE[j]; v = ds["valid"][i]; a.imshow(ds["rgb"][i]); a.axis("off"); a.set_title(f"{ds['season'][i]}  RMSE {m.rmse_px[j]:.1f}px")
    for b in range(2): a.plot(ds["rows"][i, b][v], ys[v], "g-"); a.plot(pred[j, b], ys, "y--")
plt.show(); m.agg(["mean", "median", "std"]).T.round(3)

## Final API: RGB(-D) PNG → 3D skeleton

In [ ]:
from treeskel.data import discover_samples, acquire_data
rec = discover_samples(acquire_data(cfg), cfg)[0]
r = extract_skeleton_3d(rec["rgb"], rec["depth"], model=out["model"], out_dir="outputs/predictions")
print({k: v for k, v in r.items() if k in ("length_m", "junction_row", "extent", "used_depth")})
if r["points_3d"] is not None:
    fig = plt.figure(figsize=(5, 4)); ax = fig.add_subplot(projection="3d")
    for b in ("left", "right"): P = r["points_3d"][b]; ax.plot(P[:, 0], P[:, 2], -P[:, 1], lw=3, label=b)
    ax.set_xlabel("X (m)"); ax.set_ylabel("Z (m)"); ax.set_zlabel("height (m)"); ax.legend(); plt.show()

## Report experiments (tables for the written report)
Runs SVD analysis, learning curve, season experiment, interpolation over-fitting, quadrature/ODE verification, ODE reconstruction, resolution & conditioning studies and writes them to `outputs/experiments/`.

In [ ]:
import subprocess, sys
cmd = [sys.executable, "scripts/run_experiments.py", "--data-mode", cfg["DATA_MODE"], "--out-dir", "outputs"]
if cfg["QUICK_TEST"]: cmd.append("--quick")
if not cfg["RUN_CNN"]: cmd.append("--no-cnn")
if cfg.get("LOCAL_PATH"): cmd += ["--local-path", cfg["LOCAL_PATH"]]
subprocess.run(cmd, check=True)

## Commit results from Colab
```python
from getpass import getpass
token = getpass("GitHub token (fine-grained, contents:write): ")      # never hard-code tokens
!git config user.email "you@example.com" && git config user.name "Your Name"
!git add -A && git commit -m "Update results" && git push https://{token}@github.com/abirkbexk/tree-skeleton-3d.git
```